# Airline Sentiment Lstm

Tweet tokenization, embeddings, LSTM sequence classification and three sentiment labels.

Stage: prototype.


In [ ]:
import pandas as pd
import numpy as np
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional

from google.colab import drive

In [ ]:
data = pd.read_csv("/content/Tweets.csv")

print(data.columns)

In [ ]:
# 3 sentiments -  positive, negative, neutral
data_sentiments = data[['airline_sentiment']]
print(data_sentiments)
data_sentiments.head(10)


In [ ]:
data_text_sentiment = data[['text','airline_sentiment']]
print(data_text_sentiment)

In [ ]:
data_text_sentiment["airline_sentiment"].value_counts()

In [ ]:
sentiment_label = data_text_sentiment.airline_sentiment.factorize()
print(data_sentiments)
print(sentiment_label)
# 0 neutral
# 1 positive
# 2 negative

In [ ]:
tweet = data_text_sentiment.text.values
print(tweet)

In [ ]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(tweet)
vocab_size = len(tokenizer.word_index) + 1
print(vocab_size)

In [ ]:
encoded_docs = tokenizer.texts_to_sequences(tweet)
print(encoded_docs)

In [ ]:
ml = 50
padded_sequence = pad_sequences(encoded_docs, maxlen=ml)
print(padded_sequence)

In [ ]:
embedding_vector_length = 32
model = Sequential()
model.add(Embedding(vocab_size, embedding_vector_length, input_length=ml))
model.add(LSTM(50, dropout=0.5, recurrent_dropout=0.5))
model.add(Dropout(0.2))
model.add(Dense(len(sentiment_label[1]), activation='softmax'))
model.compile(loss='sparse_categorical_crossentropy',optimizer='adam', metrics=['accuracy'])
print(model.summary())

In [ ]:
model.fit(padded_sequence,sentiment_label[0],validation_split=0.2, epochs=5, batch_size=32)

In [ ]:
def predict_sentiment(text):
    tw = tokenizer.texts_to_sequences([text])
    print(tw)
    tw = pad_sequences(tw,maxlen=ml)
    print(tw)
    prediction = int(np.argmax(model.predict(tw), axis=-1)[0])
    print("Predicted label: ", sentiment_label[1][prediction])

test_sentence1 = "I enjoyed my journey on this flight."
predict_sentiment(test_sentence1)
test_sentence2 = "This is the worst flight experience of my life!"
predict_sentiment(test_sentence2)